# Projet 1 · Analyse d'un dataset de ton choix · ⭐

**Bloc 1 · Niveau ⭐ Débutant** · Ton premier projet de portfolio.

Tu choisis un vrai dataset publié sur **Kaggle** (jeux vidéo, Netflix, Spotify ou Pokémon), tu l'explores avec pandas, puis tu poses **3 questions** et tu y réponds avec **3 graphiques**. À la séance 3, ce notebook part sur ton GitHub avec un README.

Comment travailler :
- Ce notebook tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`.
- Les cellules « À toi » sont à compléter ; elles s'exécutent déjà sans erreur avec un exemple, à toi de le remplacer par tes idées.
- Lis le README du projet pour le dataset, les étapes, le livrable et les critères de réussite.


## Préparation : choisir et charger son dataset

Les 4 datasets ont une page Kaggle (pour lire la description et voir les notebooks des autres) **et** un miroir public : le notebook charge le miroir, donc pas besoin de compte Kaggle pour commencer. Change la variable `DATASET` et relance la cellule.

| `DATASET` | Contenu | Page Kaggle |
|---|---|---|
| `"jeux_video"` | 26 688 jeux Steam : prix, propriétaires, temps de jeu, metascore | [gregorut/videogamesales](https://www.kaggle.com/datasets/gregorut/videogamesales) (les colonnes du fichier Kaggle diffèrent un peu du miroir, voir README) |
| `"netflix"` | 7 787 films et séries : type, pays, année, genres | [shivamb/netflix-shows](https://www.kaggle.com/datasets/shivamb/netflix-shows) |
| `"spotify"` | 32 833 morceaux : popularité, genre, danceability, energy... | [joebeachcapital/30000-spotify-songs](https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs) |
| `"pokemon"` | 800 Pokémon : types, PV, attaque, vitesse, légendaire | [abcsds/pokemon](https://www.kaggle.com/datasets/abcsds/pokemon) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATASET = "jeux_video"   # ← choisis : "jeux_video", "netflix", "spotify" ou "pokemon"
FICHIER_LOCAL = None     # ← ex. "vgsales.csv" si tu as déposé le fichier Kaggle dans Colab (sinon laisse None)

DATASETS = {
    "jeux_video": {
        "nom": "Jeux vidéo sur Steam (prix, joueurs, temps de jeu, notes)",
        "kaggle": "https://www.kaggle.com/datasets/gregorut/videogamesales",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2019/2019-07-30/video_games.csv",
    },
    "netflix": {
        "nom": "Catalogue Netflix (films et séries)",
        "kaggle": "https://www.kaggle.com/datasets/shivamb/netflix-shows",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv",
    },
    "spotify": {
        "nom": "30 000 morceaux Spotify (popularité, genre, caractéristiques audio)",
        "kaggle": "https://www.kaggle.com/datasets/joebeachcapital/30000-spotify-songs",
        "miroir": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv",
    },
    "pokemon": {
        "nom": "800 Pokémon (types, statistiques, génération)",
        "kaggle": "https://www.kaggle.com/datasets/abcsds/pokemon",
        "miroir": "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv",
    },
}
info = DATASETS[DATASET]

try:
    if FICHIER_LOCAL:
        df = pd.read_csv(FICHIER_LOCAL)
        print("Fichier local chargé :", FICHIER_LOCAL)
    else:
        df = pd.read_csv(info["miroir"])
        print("Miroir public chargé (mêmes données que la page Kaggle)")
except Exception as erreur:
    print("Impossible de charger les données : pas de réseau ?", erreur)
    raise

print("Dataset :", info["nom"])
print("Page Kaggle :", info["kaggle"])
print(df.shape[0], "lignes ×", df.shape[1], "colonnes")
df.head()

## 1. Découvrir le dataset (10 min)

Avant de poser des questions, on regarde ce qu'on a : combien de lignes, quelles colonnes, quel type de valeurs, où sont les cases vides. C'est le réflexe de la séance 2.

In [ ]:
print("Colonnes et types :")
print(df.dtypes)
print()
print("Cases vides par colonne :")
print(df.isna().sum())

In [ ]:
df.describe().round(1)     # résumé des colonnes numériques (moyenne, min, max...)

**À toi** : affiche 5 lignes au hasard avec `df.sample(5)`, puis note ci-dessous en une phrase ce que représente **une ligne** du dataset (un jeu ? un film ? un morceau ?).

In [ ]:
# À toi
df.sample(5, random_state=1)

In [ ]:
UNE_LIGNE = "Une ligne = un jeu vidéo vendu sur Steam"   # ← remplace par ta phrase
print(UNE_LIGNE)

## 2. Préparer quelques colonnes utiles (10 min)

Les données brutes ont souvent des colonnes texte qu'il faut transformer avant de compter ou de tracer : une date → une année, `"10,000,000 .. 20,000,000"` → un nombre, `"93 min"` → 93. La fonction ci-dessous le fait pour le dataset choisi et fixe les colonnes de l'exemple guidé (`CONFIG`), que tu peux changer.

In [ ]:
def preparer(df, dataset):
    """Ajoute quelques colonnes pratiques (année, nombres extraits du texte) selon le dataset."""
    df = df.copy()
    if dataset == "jeux_video":
        df["annee"] = pd.to_datetime(df["release_date"], errors="coerce").dt.year
        # "10,000,000 .. 20,000,000" → on garde le bas de la fourchette, en nombre
        df["proprietaires_min"] = df["owners"].str.extract(r"^([\d,]+)")[0].str.replace(",", "").astype(float)
    elif dataset == "netflix":
        df["annee_ajout"] = pd.to_datetime(df["date_added"], errors="coerce").dt.year
        df["genre_principal"] = df["listed_in"].str.split(",").str[0].str.strip()
        # "93 min" → 93 (les séries, en "Seasons", donnent une case vide)
        df["duree_min"] = df["duration"].str.extract(r"(\d+) min")[0].astype(float)
    elif dataset == "spotify":
        df["annee"] = df["track_album_release_date"].str[:4].astype(int)
        df["duree_min"] = df["duration_ms"] / 60000
    elif dataset == "pokemon":
        df = df.rename(columns={"#": "numero"})
    return df

df = preparer(df, DATASET)

# Les colonnes qu'on utilisera dans les exemples guidés (tu peux en changer !)
CONFIG = {
    "jeux_video": {"categorie": "publisher",      "valeur": "price",            "temps": "annee"},
    "netflix":    {"categorie": "genre_principal", "valeur": "duree_min",        "temps": "release_year"},
    "spotify":    {"categorie": "playlist_genre",  "valeur": "track_popularity", "temps": "annee"},
    "pokemon":    {"categorie": "Type 1",          "valeur": "Total",            "temps": "Generation"},
}[DATASET]
print("Colonnes de l'exemple guidé :", CONFIG)
print("Colonnes disponibles :", list(df.columns))

**À toi** : combien de valeurs différentes dans la colonne `CONFIG["categorie"]` ? Affiche les 10 plus fréquentes avec `value_counts().head(10)`.

<details><summary>Solution</summary>

```python
col = CONFIG["categorie"]
print(df[col].nunique(), "valeurs différentes")
print(df[col].value_counts().head(10))
```
</details>

In [ ]:
# À toi
col = CONFIG["categorie"]
nb_valeurs = df[col].nunique()
print(nb_valeurs, "valeurs différentes dans", col)

## 3. Trois questions, trois graphiques (45 min)

C'est le cœur du projet. Une **bonne question** se répond avec un chiffre ou un graphique : « Quel éditeur a sorti le plus de jeux ? » (oui) plutôt que « Les jeux sont-ils bien ? » (non). Voici des idées par dataset :

| Dataset | Idées de questions |
|---|---|
| jeux_video | Quels éditeurs sortent le plus de jeux ? Les jeux chers ont-ils un meilleur metascore ? Combien de sorties par an ? |
| netflix | Films ou séries, lesquels dominent ? Quels pays produisent le plus ? Quelle durée moyenne par genre ? |
| spotify | Quel genre est le plus populaire ? Les morceaux dansants sont-ils plus énergiques ? Les morceaux raccourcissent-ils avec les années ? |
| pokemon | Quel type est le plus fréquent ? Quel type a la meilleure attaque moyenne ? Les légendaires sont-ils vraiment plus forts ? |

Les 3 cellules suivantes proposent un schéma classique : **compter** une catégorie, **comparer** une moyenne par catégorie, **suivre** une évolution dans le temps. Elles fonctionnent avec `CONFIG` ; remplace les colonnes et le texte des questions par les tiens.

### Question 1 · Compter : quelles catégories reviennent le plus ?

In [ ]:
QUESTION_1 = f"Quelles sont les 10 valeurs les plus fréquentes de « {CONFIG['categorie']} » ?"   # ← reformule avec tes mots

top10 = df[CONFIG["categorie"]].value_counts().head(10)

plt.figure(figsize=(8, 4))
top10.sort_values().plot(kind="barh")
plt.title(QUESTION_1)
plt.xlabel("nombre de lignes")
plt.tight_layout()
plt.show()

REPONSE_1 = f"La catégorie la plus fréquente est « {top10.index[0]} » ({top10.iloc[0]} lignes)."
print(REPONSE_1)

### Question 2 · Comparer : quelle catégorie a la plus grande valeur moyenne ?

On ne garde que les catégories avec au moins 20 lignes : une moyenne sur 2 lignes ne veut rien dire (piège de la séance 5).

In [ ]:
QUESTION_2 = f"Quelle « {CONFIG['categorie']} » a la plus grande « {CONFIG['valeur']} » moyenne ?"   # ← à reformuler

cat, val = CONFIG["categorie"], CONFIG["valeur"]
frequentes = df[cat].value_counts()
frequentes = frequentes[frequentes >= 20].index
moyennes = df[df[cat].isin(frequentes)].groupby(cat)[val].mean().sort_values(ascending=False).head(10)

plt.figure(figsize=(8, 4))
moyennes.sort_values().plot(kind="barh", color="orange")
plt.title(QUESTION_2)
plt.xlabel(f"{val} (moyenne)")
plt.tight_layout()
plt.show()

REPONSE_2 = f"En moyenne, « {moyennes.index[0]} » arrive en tête avec {moyennes.iloc[0]:.1f}."
print(REPONSE_2)

### Question 3 · Suivre : comment ça évolue dans le temps ?

In [ ]:
QUESTION_3 = f"Combien de lignes par « {CONFIG['temps']} » ?"   # ← à reformuler

par_annee = df[CONFIG["temps"]].value_counts().sort_index()
par_annee = par_annee[par_annee.index >= 1980] if par_annee.index.max() > 100 else par_annee   # on coupe les années trop anciennes

plt.figure(figsize=(8, 4))
par_annee.plot(marker="o")
plt.title(QUESTION_3)
plt.xlabel(CONFIG["temps"])
plt.ylabel("nombre de lignes")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

REPONSE_3 = f"Le maximum est atteint en {par_annee.idxmax()} avec {par_annee.max()} lignes."
print(REPONSE_3)

**À toi** : maintenant une **4e question à toi**, sans schéma imposé. Idées : un nuage de points entre deux colonnes numériques (`plt.scatter`), un histogramme (`df[col].hist()`), une comparaison entre deux sous-groupes (`df[df[cat] == "X"]`).

<details><summary>Indice</summary>

```python
# Exemple : deux colonnes numériques l'une contre l'autre
num = df.select_dtypes("number").columns
plt.scatter(df[num[0]], df[num[1]], alpha=0.3)
plt.xlabel(num[0]); plt.ylabel(num[1])
```
</details>

In [ ]:
# À toi : ta 4e question (facultative mais recommandée)
QUESTION_4 = ""
REPONSE_4 = ""

## 4. Raconter : le README de ton projet (15 min)

Sur GitHub, le README est la première chose qu'on lit. La cellule ci-dessous génère un gabarit à partir de tes questions et réponses : copie le résultat dans un fichier `README.md` à côté de ton notebook (voir « Publier sur GitHub » dans le README du dossier `projets/`).

In [ ]:
TITRE = f"Analyse du dataset {info['nom']}"
AUTEUR = "Prénom"                                     # ← toi

readme = f"""# {TITRE}

Projet 1 de l'atelier Data & IA · {AUTEUR}

## Le dataset
- Source : {info['kaggle']}
- {df.shape[0]} lignes × {df.shape[1]} colonnes. {UNE_LIGNE}.

## Mes 3 questions
1. **{QUESTION_1}** → {REPONSE_1}
2. **{QUESTION_2}** → {REPONSE_2}
3. **{QUESTION_3}** → {REPONSE_3}
{('4. **' + QUESTION_4 + '** → ' + REPONSE_4) if QUESTION_4 else ''}

## Ce que j'ai appris
- (une chose sur les données)
- (une chose sur pandas / matplotlib)

## Pour lancer le notebook
Ouvrir `projet_1_analyse.ipynb` dans Google Colab et exécuter toutes les cellules : les données se chargent automatiquement.
"""
print(readme)

## 5. Auto-vérification

Avant de rendre, lance cette cellule : tout doit être ✅.

In [ ]:
def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans arrêter le notebook."""
    print(("✅ " if condition else "❌ ") + nom)

verifier("Une phrase décrit ce que représente une ligne", len(UNE_LIGNE) > 15)
verifier("3 questions reformulées (elles ne contiennent plus de « » automatiques)",
         all("«" not in q for q in [QUESTION_1, QUESTION_2, QUESTION_3]))
verifier("3 réponses écrites", all(len(r) > 10 for r in [REPONSE_1, REPONSE_2, REPONSE_3]))
verifier("Le nom de l'auteur est rempli", AUTEUR != "Prénom")
verifier("Bonus : une 4e question", bool(QUESTION_4))

## Pour aller plus loin
- Refais les 3 questions sur un **deuxième dataset** : ton code marche-t-il encore en changeant seulement `DATASET` ?
- Télécharge la vraie version Kaggle (bouton *Download*), dépose le fichier dans Colab et charge-le avec `FICHIER_LOCAL` : les colonnes ont-elles le même nom ?
- Regarde un notebook Kaggle de référence sur ton dataset (liens dans le README) et repère **une** idée de graphique à reproduire.